# 📈 Day 01a: Scalability — Horizontal vs Vertical & Beyond

---

## 🎯 What You'll Master Today
- Horizontal vs vertical scaling
- Stateless vs stateful services
- Database scaling strategies
- Microservices vs monolith

**Goal:** Confidently discuss scaling a system from 100 to 1M users.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  SCALABILITY — Core Concept                                      ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Vertical Scaling (Scale Up):                                    ║
║    Add more CPU/RAM/Disk to ONE machine                         ║
║    ✓ Simple, no code changes                                    ║
║    ✗ Hardware ceiling, single point of failure                   ║
║                                                                   ║
║  Horizontal Scaling (Scale Out):                                 ║
║    Add MORE machines behind a load balancer                     ║
║    ✓ No ceiling, fault tolerant                                 ║
║    ✗ Complexity (state, consistency, networking)                 ║
║                                                                   ║
║           ┌─────────┐                                            ║
║           │  Users  │                                            ║
║           └────┬────┘                                            ║
║                │                                                  ║
║        ┌───────┴───────┐                                         ║
║        │ Load Balancer │                                         ║
║        └───┬───┬───┬───┘                                         ║
║            │   │   │                                              ║
║         ┌──┘   │   └──┐                                          ║
║      [App1] [App2] [App3]  ← Horizontally scaled               ║
║         └──┐   │   ┌──┘                                          ║
║            │   │   │                                              ║
║        ┌───┴───┴───┴───┐                                         ║
║        │   Database    │                                         ║
║        └───────────────┘                                         ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. Scaling Journey — From 100 to 1M Users
# ============================================================

scaling_stages = {
    "Stage 1: Single Server": {
        "users": "~100",
        "architecture": "Web + App + DB on one machine",
        "bottleneck": "Single point of failure, limited resources",
    },
    "Stage 2: Separate DB": {
        "users": "~1,000",
        "architecture": "Web/App server + separate DB server",
        "bottleneck": "App server CPU/memory",
    },
    "Stage 3: Load Balancer + Multiple Servers": {
        "users": "~10,000",
        "architecture": "LB → N app servers → DB",
        "bottleneck": "Database becomes bottleneck",
        "key_requirement": "Apps must be STATELESS",
    },
    "Stage 4: DB Read Replicas + Cache": {
        "users": "~100,000",
        "architecture": "LB → Apps → Cache (Redis) → DB (master + replicas)",
        "bottleneck": "Write throughput on single master",
    },
    "Stage 5: Sharding + CDN + Message Queues": {
        "users": "~1,000,000+",
        "architecture": "CDN + LB → Apps → Cache → Sharded DB + Message Queue",
        "bottleneck": "Distributed system complexity",
    },
}

for stage, details in scaling_stages.items():
    print(f"\n{'='*60}")
    print(f"  {stage}")
    print(f"{'='*60}")
    for k, v in details.items():
        print(f"  {k}: {v}")

In [ ]:
# ============================================================
# 2. Stateless vs Stateful Services
# ============================================================

print("""
╔═══════════════════════════════════════════════════════════════╗
║  STATELESS vs STATEFUL                                       ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  STATEFUL server:                                            ║
║    Stores user session data in server memory                 ║
║    Problem: User must always hit SAME server (sticky session)║
║    → Can't scale horizontally (server goes down = data lost) ║
║                                                               ║
║  STATELESS server:                                           ║
║    No session state on server                                ║
║    State stored externally: Redis, DB, JWT token             ║
║    → ANY server can handle ANY request                       ║
║    → Easy horizontal scaling                                 ║
║                                                               ║
║  Rule: ALWAYS make your web tier stateless                   ║
║                                                               ║
║  Where to put state:                                         ║
║    • Session data → Redis / Memcached                       ║
║    • Auth → JWT tokens (client-side)                        ║
║    • Files → Object storage (S3)                            ║
║    • Persistent data → Database                             ║
║                                                               ║
╚═══════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ============================================================
# 3. Database Scaling Strategies
# ============================================================

print("""
DATABASE SCALING
════════════════

1. READ REPLICAS (most common first step)
   ┌──────────┐     ┌──────────┐
   │  Master  │────▶│ Replica1 │ ← Reads
   │ (Writes) │────▶│ Replica2 │ ← Reads
   └──────────┘     └──────────┘
   • Master handles writes, replicas handle reads
   • Works great for read-heavy apps (most apps are 80-90% reads)
   • Replication lag = eventual consistency

2. VERTICAL PARTITIONING (by table/feature)
   • User data → DB1, Product data → DB2, Orders → DB3
   • Simple to implement
   • Joins across databases become expensive

3. HORIZONTAL SHARDING (by row)
   • Split rows across multiple databases
   • Shard key: user_id % N, geographic region, hash
   • Most complex but most scalable
   • Watch for: hotspots, cross-shard queries, resharding

4. CACHING LAYER
   • Redis/Memcached in front of DB
   • Cache-aside: App checks cache → miss → query DB → store in cache
   • Reduces DB load by 80-90%
""")

In [ ]:
# ============================================================
# 4. Monolith vs Microservices
# ============================================================

comparison = [
    ("Aspect", "Monolith", "Microservices"),
    ("Deployment", "All-or-nothing", "Independent per service"),
    ("Scaling", "Scale entire app", "Scale individual services"),
    ("Tech stack", "Single stack", "Polyglot (mix languages)"),
    ("Data", "Shared database", "Database per service"),
    ("Complexity", "Simple initially", "Complex (networking, discovery)"),
    ("Testing", "Easier E2E", "Complex integration tests"),
    ("Team size", "Small teams", "Large orgs (2-pizza teams)"),
    ("Failure", "Entire app down", "Partial failure (resilient)"),
]

# Print as table
for row in comparison:
    print(f"  {row[0]:<15} │ {row[1]:<25} │ {row[2]}")
    if row[0] == "Aspect":
        print(f"  {'─'*15}─┼─{'─'*25}─┼─{'─'*30}")

print("""
💡 Interview Answer Framework:
   "Start with a monolith. Break into microservices ONLY when:"
   • Team grows beyond 2-3 squads
   • Different parts need independent scaling
   • Different parts need different tech stacks
   • Deployment conflicts become frequent
""")

In [ ]:
# ============================================================
# 5. Scalability Checklist
# ============================================================

print("""
╔═══════════════════════════════════════════════════════════════╗
║  SCALABILITY CHECKLIST (use in interviews)                   ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  □ Stateless web tier (session in Redis/JWT)                 ║
║  □ Load balancer in front                                    ║
║  □ CDN for static assets                                     ║
║  □ Database read replicas                                    ║
║  □ Cache layer (Redis) for hot data                          ║
║  □ Message queue for async tasks                             ║
║  □ Database sharding if needed                               ║
║  □ Monitoring & auto-scaling                                 ║
║                                                               ║
║  Scale in THIS order:                                        ║
║  1. Optimize code & queries (free!)                          ║
║  2. Add caching (biggest bang for buck)                      ║
║  3. Read replicas                                            ║
║  4. Horizontal scaling of app tier                           ║
║  5. Sharding (last resort — most complex)                   ║
║                                                               ║
╚═══════════════════════════════════════════════════════════════╝
""")

---

## 🗺️ Scalability Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  SCALABILITY ESSENTIALS                                          ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Vertical = bigger machine. Horizontal = more machines.         ║
║  Horizontal requires STATELESS services.                        ║
║                                                                   ║
║  DB scaling: replicas → partitioning → sharding                ║
║  App scaling: LB + N stateless servers                          ║
║  Cache: Redis/Memcached (80-90% read reduction)                 ║
║                                                                   ║
║  Monolith first, microservices when needed.                     ║
║  Scale order: optimize → cache → replicas → scale out → shard ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Horizontal vs vertical scaling? | Vertical=bigger machine (ceiling, SPOF). Horizontal=more machines (needs stateless, LB) |
| 2 | How to scale from 100 to 1M users? | Separate DB → LB + stateless servers → cache + replicas → sharding + CDN + queues |
| 3 | Why stateless web tier? | Any server handles any request. Easy to add/remove servers. State goes to Redis/JWT/DB |
| 4 | Monolith vs microservices? | Start monolith. Micro when: independent scaling, large teams, different stacks needed |
| 5 | What to scale first? | Optimize code → add cache → read replicas → horizontal app scaling → DB sharding |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Horizontal scaling = add machines (needs stateless)  │
## │  • Stateless web tier is NON-NEGOTIABLE                  │
## │  • Cache (Redis) = biggest ROI in scaling                │
## │  • Start monolith, break into micro when needed          │
## │  • DB scaling: replicas → partitioning → sharding       │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **CAP Theorem & Consistency** — trade-offs in distributed systems